# 08 — อ่าน–เขียนไฟล์ และการจัดการ Error

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**วันที่ 5 · ช่วงที่ 1 · เวลาโดยประมาณ 60 นาที**

---

### สิ่งที่จะได้จากบทนี้
- เปิด/อ่าน/เขียนไฟล์ข้อความด้วย `with open(...)`
- อ่านไฟล์ **CSV** ด้วยมือ เพื่อเข้าใจว่า pandas ทำอะไรให้เราจริง ๆ
- อัปโหลด/ดาวน์โหลดไฟล์ใน Colab และ mount Google Drive
- ใช้ `try` / `except` จัดการข้อมูลสกปรกโดยโปรแกรมไม่ล้ม

> **วิธีเรียน:** โค้ดในเซลล์สีเทาคือโค้ดที่อาจารย์จะรันให้ดู
> ใต้แต่ละหัวข้อจะมีเซลล์ ✍️ ว่างไว้ — **ให้พิมพ์ตามด้วยตัวเอง ห้าม copy–paste**
> การพิมพ์เองคือวิธีเดียวที่จะจำ syntax ได้

## 0. เตรียมไฟล์ข้อมูล

รันเซลล์นี้ก่อนเสมอ (รันครั้งเดียวต่อ session) จะได้โฟลเดอร์ `data/` มา 3 ไฟล์

In [ ]:
# ===== เตรียมไฟล์ข้อมูลสำหรับบทนี้ (รันครั้งเดียวต่อ session) =====
# ถ้าอาจารย์เปลี่ยนชื่อ repo ให้แก้ URL บรรทัดล่างนี้บรรทัดเดียว
BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

import os, subprocess, sys, urllib.request

FILES = ["stations.csv", "rainfall_daily_2025.csv", "messy_rainfall.csv"]
os.makedirs("data", exist_ok=True)


def have_all():
    return all(os.path.exists(os.path.join("data", n)) for n in FILES)


# 1) ถ้ามีไฟล์อยู่แล้ว (เช่น clone repo มา หรืออัปโหลดเอง) ก็ใช้ได้เลย
if not have_all():
    # 2) ลองดาวน์โหลดจาก GitHub
    for name in FILES:
        target = os.path.join("data", name)
        if not os.path.exists(target):
            try:
                urllib.request.urlretrieve(BASE + "data/" + name, target)
            except Exception:
                pass

if not have_all():
    # 3) สำรองสุดท้าย: สร้างข้อมูลชุดเดียวกันขึ้นมาใหม่จาก make_datasets.py
    script = None
    for cand in ["make_datasets.py", "utils/make_datasets.py"]:
        if os.path.exists(cand):
            script = cand
            break
    if script is None:
        try:
            urllib.request.urlretrieve(BASE + "utils/make_datasets.py", "make_datasets.py")
            script = "make_datasets.py"
        except Exception:
            script = None
    if script:
        subprocess.run([sys.executable, script, "--outdir", "data"], check=True)

if have_all():
    print("ข้อมูลพร้อมใช้งาน:", sorted(os.listdir("data")))
else:
    print("ยังไม่พบข้อมูล -> ให้ upload โฟลเดอร์ data/ เข้ามา "
          "หรือรัน utils/make_datasets.py เพื่อสร้างข้อมูลใหม่")

## 1. เขียนไฟล์ด้วย `with open()`

`open(path, mode)` โดย mode ที่ใช้บ่อย: `"r"` อ่าน, `"w"` เขียนทับ, `"a"` เขียนต่อท้าย

ใช้ `with` เสมอ เพราะมันปิดไฟล์ให้อัตโนมัติแม้โปรแกรมจะ error กลางทาง
`encoding="utf-8"` จำเป็นเมื่อมีภาษาไทย

In [ ]:
with open("note.txt", "w", encoding="utf-8") as f:
    f.write("บันทึกการตรวจอากาศ\n")
    f.write("สถานี CHB01\n")
    f.write("ฝน 45.5 mm\n")

print("เขียนไฟล์เรียบร้อย")
!ls -l note.txt

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 2. อ่านไฟล์ 3 แบบ

In [ ]:
# แบบที่ 1: อ่านทั้งไฟล์เป็นข้อความเดียว
with open("note.txt", encoding="utf-8") as f:
    print(f.read())

# แบบที่ 2: อ่านเป็น list ของบรรทัด
with open("note.txt", encoding="utf-8") as f:
    lines = f.readlines()
print(lines)

# แบบที่ 3 (แนะนำ): วนทีละบรรทัด ประหยัดหน่วยความจำ ใช้ได้กับไฟล์ใหญ่มาก
with open("note.txt", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        print(i, line.strip())

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 3. อ่าน CSV ด้วยมือ

ไฟล์ CSV คือไฟล์ข้อความธรรมดาที่คั่นด้วยจุลภาค บรรทัดแรกคือ **header**
เรามาลองอ่านเองก่อน เพื่อให้เห็นว่า pandas ในบทหน้าทำงานแทนเรากี่ขั้นตอน

In [ ]:
path = "data/rainfall_daily_2025.csv"

with open(path, encoding="utf-8-sig") as f:
    header = f.readline().strip().split(",")
    print("คอลัมน์:", header)

    total = 0.0
    count = 0
    for line in f:
        fields = line.strip().split(",")
        if fields[1] != "CHB01":       # เอาเฉพาะสถานี CHB01
            continue
        total += float(fields[4])
        count += 1

print(f"CHB01: {count} วัน ฝนรวม {total:.1f} mm")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ใช้ module `csv` ช่วยอ่านให้ปลอดภัยกว่า

`csv.DictReader` จะจับคู่ header กับค่าให้เลย เรียกด้วยชื่อคอลัมน์ได้เหมือน dictionary
(และจัดการกรณีข้อมูลมีจุลภาคในเครื่องหมายคำพูดได้ถูกต้อง ซึ่ง `split(",")` ทำไม่ได้)

In [ ]:
import csv

with open("data/stations.csv", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['station_id']} {row['station_name']:<10} "
              f"({row['lat']}, {row['lon']}) สูง {row['elevation_m']} m")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### เขียนผลลัพธ์ออกเป็น CSV

In [ ]:
summary = [
    {"station_id": "CHB01", "total_mm": 1080.5},
    {"station_id": "SRC01", "total_mm": 946.6},
    {"station_id": "NYI01", "total_mm": 1435.8},
]

with open("annual_summary.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["station_id", "total_mm"])
    writer.writeheader()
    writer.writerows(summary)

!cat annual_summary.csv

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4. ไฟล์ใน Colab: อัปโหลด ดาวน์โหลด และ Google Drive

- ไฟล์ที่สร้างในเซสชันจะ **หายไป** เมื่อ runtime ถูกตัด → งานสำคัญต้องเซฟลง Drive
- ดูไฟล์ทั้งหมดได้จากไอคอนโฟลเดอร์ 📁 ทางซ้ายมือ

```python
# อัปโหลดไฟล์จากเครื่องตัวเอง
from google.colab import files
uploaded = files.upload()

# ดาวน์โหลดไฟล์ผลลัพธ์กลับเครื่อง
files.download("annual_summary.csv")

# ต่อกับ Google Drive (จะมีหน้าต่างให้อนุญาตสิทธิ์)
from google.colab import drive
drive.mount("/content/drive")
# แล้วเซฟไว้ที่ /content/drive/MyDrive/...
```
> เซลล์ข้างบนเป็นตัวอย่าง ให้คัดลอกไปรันใน cell ใหม่เมื่อต้องใช้จริง
> เพราะ `files.upload()` จะค้างรอไฟล์ ทำให้รันทั้ง notebook รวดเดียวไม่ได้

## 5. Error handling: `try` / `except`

ข้อมูลจริงไม่เคยสะอาด ลองดูไฟล์ `messy_rainfall.csv` — มีทั้งช่องว่าง, `NA`, `-999`, ภาษาไทย
ถ้าเราสั่ง `float()` ตรง ๆ โปรแกรมจะหยุดกลางทาง

In [ ]:
import csv

with open("data/messy_rainfall.csv", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

print(len(rows), "แถว")
for r in rows[:8]:
    print(r["date"], "|", repr(r["rain_mm"]))

# ค่าที่แปลงเป็นตัวเลขไม่ได้ มีอะไรบ้าง
weird = {r["rain_mm"] for r in rows if not r["rain_mm"].strip().replace(".", "").replace("-", "").isdigit()}
print("ค่าแปลก ๆ:", weird)

In [ ]:
# แบบไม่ป้องกัน -> พังกลางทาง
total = 0
for r in rows:
    total += float(r["rain_mm"])

### โครงสร้าง `try` / `except` / `else` / `finally`

```python
try:
    โค้ดที่อาจพัง
except ชนิดของ Error:
    ทำอะไรเมื่อพัง
else:
    ทำเมื่อไม่พัง        (ไม่จำเป็นต้องมี)
finally:
    ทำทุกกรณี            (ไม่จำเป็นต้องมี)
```

| Error ที่เจอบ่อย | สาเหตุ |
|---|---|
| `ValueError` | แปลงชนิดข้อมูลไม่ได้ เช่น `float("NA")` |
| `KeyError` | เรียก key ที่ไม่มีใน dict |
| `IndexError` | เรียกตำแหน่งเกินขนาด list |
| `FileNotFoundError` | ไม่พบไฟล์ |
| `ZeroDivisionError` | หารด้วยศูนย์ |
| `TypeError` | ใช้ชนิดข้อมูลผิด เช่น `"5" + 5` |

In [ ]:
MISSING_FLAGS = (-999, -9999)

total = 0.0
good = 0
bad = 0
bad_values = []

for r in rows:
    raw = r["rain_mm"]
    try:
        value = float(raw.strip())
    except ValueError:
        bad += 1
        bad_values.append(raw)
        continue

    if value in MISSING_FLAGS:
        bad += 1
        bad_values.append(raw)
        continue

    total += value
    good += 1

print(f"ใช้ได้ {good} แถว | เสีย {bad} แถว")
print(f"ฝนรวม {total:.1f} mm | เฉลี่ย {total/good:.2f} mm/วัน")
print("ค่าที่ทิ้ง:", bad_values)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### `raise` — สร้าง error ของเราเอง

เมื่อค่าที่รับเข้ามาไม่สมเหตุสมผล การหยุดพร้อมข้อความชัด ๆ ดีกว่าปล่อยให้คำนวณผิดเงียบ ๆ

In [ ]:
def check_latitude(lat):
    """ตรวจว่าค่า latitude อยู่ในช่วงที่เป็นไปได้"""
    if not (-90 <= lat <= 90):
        raise ValueError(f"latitude ต้องอยู่ระหว่าง -90 ถึง 90 แต่ได้ {lat}")
    return lat

print(check_latitude(13.367))

try:
    check_latitude(133.67)          # พิมพ์จุดผิดตำแหน่ง
except ValueError as e:
    print("ตรวจพบข้อผิดพลาด:", e)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 6. แบบฝึกหัดท้ายบท

1. เขียนฟังก์ชัน `to_number(value)` ที่แปลงข้อความเป็น float คืน `None` เมื่อแปลงไม่ได้
   หรือเป็นค่า `-999` แล้วทดสอบกับ `["12.5", " 8.4 ", "NA", "", "-999", "ไม่มีข้อมูล"]`
2. ใช้ `to_number()` อ่าน `messy_rainfall.csv` แล้วสรุป: จำนวนแถวทั้งหมด, แถวที่ใช้ได้,
   ฝนรวม, ฝนสูงสุด และวันที่ของฝนสูงสุด
3. เขียนผลลัพธ์ที่ทำความสะอาดแล้วออกเป็นไฟล์ `clean_rainfall.csv` (ข้ามแถวที่เสีย)
4. ลองเปิดไฟล์ที่ไม่มีอยู่จริง แล้วดักด้วย `except FileNotFoundError` พร้อมพิมพ์ข้อความที่เป็นมิตร

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3

In [ ]:
# ข้อ 4

### เฉลย
```python
# ข้อ 1
def to_number(value):
    """แปลงข้อความเป็น float คืน None เมื่อใช้ไม่ได้"""
    try:
        n = float(str(value).strip())
    except ValueError:
        return None
    return None if n in (-999, -9999) else n

for v in ["12.5", " 8.4 ", "NA", "", "-999", "ไม่มีข้อมูล"]:
    print(repr(v), "->", to_number(v))

# ข้อ 2
import csv
with open("data/messy_rainfall.csv", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

clean = [(r["date"], to_number(r["rain_mm"])) for r in rows]
usable = [(d, v) for d, v in clean if v is not None]
best_date, best_val = max(usable, key=lambda t: t[1])
print(f"ทั้งหมด {len(rows)} | ใช้ได้ {len(usable)}")
print(f"ฝนรวม {sum(v for _, v in usable):.1f} mm")
print(f"สูงสุด {best_val:.1f} mm วันที่ {best_date}")

# ข้อ 3
with open("clean_rainfall.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["date", "rain_mm"])
    w.writerows(usable)
print("เขียนไฟล์แล้ว")

# ข้อ 4
try:
    with open("data/no_such_file.csv") as f:
        f.read()
except FileNotFoundError:
    print("ไม่พบไฟล์ที่ระบุ กรุณาตรวจสอบชื่อไฟล์และโฟลเดอร์อีกครั้ง")
```